<a href="https://colab.research.google.com/github/Frans1828/scikit-learn-cookbook/blob/main/03.%20Dimensionality%20Reduction/ch03_dimensionality_reduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## Identitas Mahasiswa

| Informasi | Keterangan |
|---|---|
| **Nama** | Fransisco Sitepu |
| **NIM** | 101032300004 |
| **Kelas** | TK 47 03 |
| **Mata Kuliah** | Machine Learning |
| **Institusi** | Telkom University |

---


# Chapter 3: Dimensionality Reduction

## Curse of Dimensionality

Ketika jumlah fitur $p$ sangat besar, beberapa masalah muncul:
- **Sparsity**: Data menjadi sangat jarang di ruang dimensi tinggi
- **Overfitting**: Model belajar noise daripada pola nyata
- **Computational Cost**: Meningkat eksponensial
- **Visualization**: Tidak mungkin visualisasi $> 3$ dimensi

Dimensionality reduction mengatasi masalah ini dengan memproyeksikan data ke ruang berdimensi lebih rendah sambil **mempertahankan informasi penting**.

### Dua Pendekatan Utama
1. **Feature Extraction**: Membuat fitur baru (kombinasi dari fitur asli) — PCA, LDA, t-SNE
2. **Feature Selection**: Memilih subset dari fitur asli — SelectKBest, RFE, VarianceThreshold


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, load_iris, load_wine, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load datasets
digits = load_digits()   # 8x8 = 64 features, 10 classes
iris = load_iris()       # 4 features, 3 classes
wine = load_wine()       # 13 features, 3 classes
cancer = load_breast_cancer()  # 30 features, 2 classes

print(f"Digits:  X={digits.data.shape}, classes={len(digits.target_names)}")
print(f"Iris:    X={iris.data.shape}, classes={len(iris.target_names)}")
print(f"Wine:    X={wine.data.shape}, classes={len(wine.target_names)}")
print(f"Cancer:  X={cancer.data.shape}, classes={len(cancer.target_names)}")


## 3.1 Principal Component Analysis (PCA)

PCA menemukan arah (principal components) di mana variansi data maksimal. Secara matematis, PCA melakukan dekomposisi nilai eigen dari matriks kovarian:

$$\boldsymbol{\Sigma} = \frac{1}{n} \mathbf{X}^\top \mathbf{X} = \mathbf{V} \boldsymbol{\Lambda} \mathbf{V}^\top$$

di mana:
- $\mathbf{V}$ = matriks eigenvector (principal components)
- $\boldsymbol{\Lambda}$ = matriks diagonal eigenvalue

**Explained Variance Ratio** untuk komponen ke-$k$:

$$\text{EVR}_k = \frac{\lambda_k}{\sum_{i=1}^{p} \lambda_i}$$

Jumlah komponen dipilih sehingga *cumulative* EVR mencapai threshold yang diinginkan (biasanya 90-95%).


In [ ]:
from sklearn.decomposition import PCA

# Standardize the data first (important for PCA)
scaler = StandardScaler()
X_wine = scaler.fit_transform(wine.data)

# Fit PCA with all components
pca_full = PCA()
pca_full.fit(X_wine)

# Plot explained variance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Individual explained variance
axes[0].bar(range(1, len(pca_full.explained_variance_ratio_)+1),
            pca_full.explained_variance_ratio_, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Individual Explained Variance (Wine Dataset)')

# Cumulative explained variance
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
axes[1].plot(range(1, len(cumvar)+1), cumvar, 'o-', color='tomato', lw=2)
axes[1].axhline(y=0.90, color='green', linestyle='--', label='90% threshold')
axes[1].axhline(y=0.95, color='orange', linestyle='--', label='95% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()

plt.tight_layout()
plt.show()

# Find number of components for 95% variance
n_components_95 = np.argmax(cumvar >= 0.95) + 1
n_components_90 = np.argmax(cumvar >= 0.90) + 1
print(f"Components needed for 90% variance: {n_components_90}")
print(f"Components needed for 95% variance: {n_components_95}")
print(f"Original features: {X_wine.shape[1]}")
print(f"Reduction ratio (95%): {n_components_95/X_wine.shape[1]:.1%}")


In [ ]:
# PCA 2D visualization and reconstruction
pca_2d = PCA(n_components=2)
X_wine_2d = pca_2d.fit_transform(X_wine)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 2D scatter plot
colors = ['#E91E63', '#2196F3', '#4CAF50']
for cls, color, name in zip([0,1,2], colors, wine.target_names):
    mask = wine.target == cls
    axes[0].scatter(X_wine_2d[mask, 0], X_wine_2d[mask, 1],
                    c=color, label=name, alpha=0.7, s=60)
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)')
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)')
axes[0].set_title('Wine Dataset: PCA 2D Projection')
axes[0].legend()

# Biplot: feature loadings
loadings = pca_2d.components_.T
for i, (loading, feature) in enumerate(zip(loadings, wine.feature_names)):
    axes[1].arrow(0, 0, loading[0]*3, loading[1]*3,
                  head_width=0.1, head_length=0.05, fc='red', ec='red')
    axes[1].text(loading[0]*3.2, loading[1]*3.2, feature, fontsize=7, ha='center')
axes[1].set_xlim(-4, 4)
axes[1].set_ylim(-4, 4)
axes[1].axhline(0, color='gray', lw=0.5)
axes[1].axvline(0, color='gray', lw=0.5)
axes[1].set_title('PCA Biplot: Feature Loadings')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')

plt.tight_layout()
plt.show()

print(f"Total variance explained by 2 PCs: {pca_2d.explained_variance_ratio_.sum():.1%}")


## 3.2 Kernel PCA

PCA standar hanya menemukan struktur **linear**. Kernel PCA menggunakan **kernel trick** untuk mengungkap struktur non-linear:

$$K(\mathbf{x}_i, \mathbf{x}_j) = \phi(\mathbf{x}_i)^\top \phi(\mathbf{x}_j)$$

Kernel yang umum:
- **RBF (Gaussian)**: $K(x, x') = \exp(-\gamma \|x-x'\|^2)$
- **Polynomial**: $K(x, x') = (\gamma x^\top x' + r)^d$


In [ ]:
from sklearn.decomposition import KernelPCA
from sklearn.datasets import make_moons, make_circles

# Create non-linearly separable data
X_circles, y_circles = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)
X_moons, y_moons = make_moons(n_samples=300, noise=0.05, random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for row, (X_data, y_data, title) in enumerate([
    (X_circles, y_circles, 'Circles'),
    (X_moons, y_moons, 'Moons')
]):
    # Original data
    for cls in [0, 1]:
        mask = y_data == cls
        axes[row, 0].scatter(X_data[mask, 0], X_data[mask, 1],
                             alpha=0.6, s=20, label=f'Class {cls}')
    axes[row, 0].set_title(f'{title}: Original')
    axes[row, 0].legend()

    # Linear PCA
    pca_lin = PCA(n_components=1)
    X_pca = pca_lin.fit_transform(X_data)
    for cls in [0, 1]:
        mask = y_data == cls
        axes[row, 1].scatter(X_pca[mask, 0], np.zeros(mask.sum()),
                             alpha=0.6, s=20, label=f'Class {cls}')
    axes[row, 1].set_title(f'{title}: Linear PCA')
    axes[row, 1].legend()

    # Kernel PCA (RBF)
    kpca = KernelPCA(n_components=2, kernel='rbf', gamma=10)
    X_kpca = kpca.fit_transform(X_data)
    for cls in [0, 1]:
        mask = y_data == cls
        axes[row, 2].scatter(X_kpca[mask, 0], X_kpca[mask, 1],
                             alpha=0.6, s=20, label=f'Class {cls}')
    axes[row, 2].set_title(f'{title}: Kernel PCA (RBF)')
    axes[row, 2].legend()

plt.tight_layout()
plt.show()


## 3.3 Linear Discriminant Analysis (LDA)

LDA adalah metode dimensionality reduction **supervised** yang memaksimalkan separabilitas antar kelas. Berbeda dengan PCA yang memaksimalkan variansi total, LDA memaksimalkan rasio:

$$J(\mathbf{w}) = \frac{\mathbf{w}^\top \mathbf{S}_B \mathbf{w}}{\mathbf{w}^\top \mathbf{S}_W \mathbf{w}}$$

di mana:
- $\mathbf{S}_B$ = **Between-class scatter matrix** (memaksimalkan ini → kelas terpisah jauh)
- $\mathbf{S}_W$ = **Within-class scatter matrix** (meminimalkan ini → kelas kompak)

LDA dapat menghasilkan maksimal $\min(K-1, p)$ komponen untuk dataset dengan $K$ kelas dan $p$ fitur.


In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

X_wine_sc = StandardScaler().fit_transform(wine.data)
y_wine = wine.target

# LDA with all components
lda = LinearDiscriminantAnalysis(n_components=2)
X_lda = lda.fit_transform(X_wine_sc, y_wine)

# PCA for comparison
pca_compare = PCA(n_components=2)
X_pca_compare = pca_compare.fit_transform(X_wine_sc)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#E91E63', '#2196F3', '#4CAF50']

for (ax, X_proj, title, xlabel, ylabel) in [
    (axes[0], X_pca_compare, 'PCA (Unsupervised)', 'PC1', 'PC2'),
    (axes[1], X_lda, 'LDA (Supervised)', 'LD1', 'LD2')
]:
    for cls, color, name in zip([0,1,2], colors, wine.target_names):
        mask = y_wine == cls
        ax.scatter(X_proj[mask, 0], X_proj[mask, 1],
                   c=color, label=name, alpha=0.8, s=60,
                   edgecolors='white', linewidth=0.5)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(f'Wine Dataset: {title}')
    ax.legend()

plt.tight_layout()
plt.show()

print(f"LDA explained variance ratio: {lda.explained_variance_ratio_}")
print(f"LDA max components possible: {min(3-1, wine.data.shape[1])} (= min(K-1, p))")


## 3.4 t-SNE: Visualisasi Non-Linear

t-SNE (t-distributed Stochastic Neighbor Embedding) adalah teknik visualisasi non-linear yang sangat populer untuk mengeksplorasi struktur tersembunyi data berdimensi tinggi.

t-SNE memodelkan kesamaan antar titik data dengan distribusi probabilitas:
- **High-dimensional space**: menggunakan distribusi Gaussian
- **Low-dimensional space**: menggunakan distribusi t (Student's t) dengan derajat kebebasan 1

Kullback-Leibler divergence diminimalkan:

$$KL(P \| Q) = \sum_i \sum_j p_{ij} \log \frac{p_{ij}}{q_{ij}}$$

**Catatan**: t-SNE adalah alat **visualisasi** bukan fitur untuk modeling. Jangan gunakan hasil t-SNE sebagai input model.


In [ ]:
from sklearn.manifold import TSNE

# Apply t-SNE to digits dataset (64 features -> 2D)
# Take subset for speed
X_digits = digits.data[:800]
y_digits = digits.target[:800]
X_digits_sc = StandardScaler().fit_transform(X_digits)

# t-SNE
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X_digits_sc)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# PCA 2D comparison
pca_digits = PCA(n_components=2)
X_pca_digits = pca_digits.fit_transform(X_digits_sc)

cmap = plt.cm.get_cmap('tab10', 10)
for digit in range(10):
    mask = y_digits == digit
    axes[0].scatter(X_pca_digits[mask, 0], X_pca_digits[mask, 1],
                    c=[cmap(digit)], label=str(digit), s=15, alpha=0.6)
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                    c=[cmap(digit)], label=str(digit), s=15, alpha=0.6)

axes[0].set_title('Digits: PCA (2D)')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].legend(title='Digit', ncol=2, fontsize=8)

axes[1].set_title('Digits: t-SNE (2D, perplexity=30)')
axes[1].set_xlabel('t-SNE 1'); axes[1].set_ylabel('t-SNE 2')
axes[1].legend(title='Digit', ncol=2, fontsize=8)

plt.tight_layout()
plt.show()


## 3.5 Feature Selection

Feature selection memilih subset fitur yang paling relevan, berbeda dengan feature extraction yang membuat fitur baru.

### Metode Utama:

| Metode | Kategori | Penjelasan |
|---|---|---|
| **VarianceThreshold** | Filter | Hapus fitur dengan variansi di bawah threshold |
| **SelectKBest** | Filter | Pilih K fitur dengan skor statistik terbaik |
| **RFE** | Wrapper | Eliminasi fitur rekursif berdasarkan bobot model |
| **SelectFromModel** | Embedded | Pilih fitur berdasarkan feature importance |


In [ ]:
from sklearn.feature_selection import (
    VarianceThreshold, SelectKBest, f_classif, mutual_info_classif,
    RFE, SelectFromModel
)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X_cancer = cancer.data
y_cancer = cancer.target

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_cancer, y_cancer, test_size=0.3, random_state=42, stratify=y_cancer
)

results = {}

# Baseline: all features
from sklearn.pipeline import Pipeline
base_pipe = Pipeline([('scaler', StandardScaler()), ('svc', SVC(random_state=42))])
base_pipe.fit(X_tr_c, y_tr_c)
results['All features (30)'] = accuracy_score(y_te_c, base_pipe.predict(X_te_c))

# SelectKBest with F-test
selector_f = SelectKBest(f_classif, k=10)
X_tr_f = selector_f.fit_transform(X_tr_c, y_tr_c)
X_te_f = selector_f.transform(X_te_c)
pipe_f = Pipeline([('scaler', StandardScaler()), ('svc', SVC(random_state=42))])
pipe_f.fit(X_tr_f, y_tr_c)
results['SelectKBest F-test (k=10)'] = accuracy_score(y_te_c, pipe_f.predict(X_te_f))

# SelectKBest with Mutual Information
selector_mi = SelectKBest(mutual_info_classif, k=10)
X_tr_mi = selector_mi.fit_transform(X_tr_c, y_tr_c)
X_te_mi = selector_mi.transform(X_te_c)
pipe_mi = Pipeline([('scaler', StandardScaler()), ('svc', SVC(random_state=42))])
pipe_mi.fit(X_tr_mi, y_tr_c)
results['SelectKBest MI (k=10)'] = accuracy_score(y_te_c, pipe_mi.predict(X_te_mi))

# RFE
from sklearn.linear_model import LogisticRegression
rfe = RFE(LogisticRegression(max_iter=5000, random_state=42), n_features_to_select=10)
X_tr_rfe = rfe.fit_transform(StandardScaler().fit_transform(X_tr_c), y_tr_c)
X_te_rfe = rfe.transform(StandardScaler().fit_transform(X_te_c))
lr_rfe = LogisticRegression(max_iter=5000, random_state=42)
lr_rfe.fit(X_tr_rfe, y_tr_c)
results['RFE (k=10)'] = accuracy_score(y_te_c, lr_rfe.predict(X_te_rfe))

# SelectFromModel (RF importance)
rf_sel = RandomForestClassifier(n_estimators=100, random_state=42)
sfm = SelectFromModel(rf_sel, max_features=10)
X_tr_sfm = sfm.fit_transform(X_tr_c, y_tr_c)
X_te_sfm = sfm.transform(X_te_c)
pipe_sfm = Pipeline([('scaler', StandardScaler()), ('svc', SVC(random_state=42))])
pipe_sfm.fit(X_tr_sfm, y_tr_c)
results['SelectFromModel RF'] = accuracy_score(y_te_c, pipe_sfm.predict(X_te_sfm))

# Display results
print("=== Feature Selection Comparison (Breast Cancer Dataset) ===")
for method, acc in results.items():
    print(f"{method:<35}: {acc:.4f}")


## Ringkasan: Key Takeaways Chapter 3

| Metode | Tipe | Supervised? | Keunggulan | Keterbatasan |
|---|---|---|---|---|
| **PCA** | Extraction | Tidak | Mempertahankan variansi maksimal; cepat | Hanya linear |
| **Kernel PCA** | Extraction | Tidak | Menangkap struktur non-linear | Tidak ada inverse transform; lambat |
| **LDA** | Extraction | Ya | Memaksimalkan separabilitas kelas | Maks K-1 komponen; asumsi Gaussian |
| **t-SNE** | Extraction | Tidak | Visualisasi cluster sangat baik | Hanya untuk visualisasi, non-deterministic |
| **SelectKBest** | Selection | Ya | Sederhana, cepat | Tidak mempertimbangkan interaksi fitur |
| **RFE** | Selection | Ya | Mempertimbangkan kontribusi relatif | Lambat untuk banyak fitur |
| **SelectFromModel** | Selection | Ya | Terintegrasi dengan model; interpretable | Bergantung pada model base |

### Panduan Pemilihan Metode
- **Exploratory visualization**: gunakan t-SNE atau PCA 2D
- **Preprocessing sebelum modeling**: PCA atau LDA (supervised)
- **Feature importance & selection**: SelectFromModel atau RFE
- **Data sangat berdimensi tinggi & sparse** (teks, genomik): TruncatedSVD
